# Jacobian 引导的谱调度

我们研究一个三层线性网络
$$
\hat A = W_3 W_2 W_1,
\qquad
\hat A \approx A.
$$
每一步训练都会为每一层计算普通梯度，但在应用更新前，我们允许改变更新矩阵的谱形状。对原始更新矩阵 $U$，定义
$$
\operatorname{spec}_p(U)
=
P\operatorname{diag}(\sigma_i^p)Q^\top
\cdot
\frac{\|U\|_F}{\|P\operatorname{diag}(\sigma_i^p)Q^\top\|_F},
\qquad
U=P\operatorname{diag}(\sigma_i)Q^\top.
$$
指数 $p$ 控制权衡方式：

- $p=1$：标准 SGD 更新。
- $p=0.5$：部分压平更新谱。
- $p=0$：使用正交化后的更新方向，并保持相同 Frobenius 范数。

每一步的动作是一个调度 $a=(p_1,p_2,p_3)\in\{1,0.5,0\}^3$，即每层一个谱指数。我们比较三种调度选择方式：

- **sgd**：始终使用 $(1,1,1)$。
- **dp_rollout**：每 $T$ 步，对每个候选调度真实模拟 $H$ 步，选择 rollout 末端 loss 最低的动作。
- **dp_jacobian**：每 $T$ 步，用候选动作的一步 loss 和基于 Jacobian 的未来收敛速度预测来打分。

目标不只是找到更好的调度，还要预测何时谱形状权衡是值得的：某个调度可能会牺牲当前一步的 loss，却改善局部几何，从而让后续优化更快。

这个视角也对应了贪心算法与动态规划的区别。贪心规则选择当前一步下降最大的动作；在这里，就是选择一步后 loss 最小的调度。动态规划则关心一个动作会把系统转移到什么状态，以及这个新状态的未来价值是否更好。我们的谱调度问题正是这种形式：一个动作不仅是在更新参数以降低当前 loss，也是在把权重转移到一个新的状态；这个状态的几何结构可能让未来优化更容易，也可能更困难。

下面的 Jacobian 价值估计，就是对这种未来价值的一个局部、可微近似。它不只是问当前 loss 是否下降最快，而是问转移后的状态是否具有更大的有效收缩率。因此，谱调度本质上是在寻找更好的状态转移，而不仅仅是在寻找当前最陡的下降方向。



## 实验设置

三种方法使用相同的目标矩阵和相同的初始权重。唯一的区别是谱调度如何选择。rollout 方法是经验规划基线；Jacobian 方法用局部价值估计替代大部分经验搜索。


In [ ]:
import math
from collections import Counter
from itertools import product as cartesian_product

import matplotlib.pyplot as plt
import torch

torch.set_default_dtype(torch.float64)

assert torch.cuda.is_available()
DEVICE = torch.device("cuda")

D = 64
L = 3
STEPS = 100
LR = 2e-2
SEED = 0

# 谱动作：离散化程度（与层数 L 共同决定动作空间大小 3^L）
ACTIONS = (1.0, 0.5, 0.0)
ALL_SCHEDULES = list(cartesian_product(ACTIONS, repeat=L))

# 通用 DP 超参（不随训练阶段、调度类型变化）
ROLLOUT_H = 15
COMMIT_T = 10
# 可选：粗筛省算力（精筛目标仍是 rollout 末端 loss）
USE_COARSE_FINE = True
ROLLOUT_H_COARSE = 5
ROLLOUT_TOP_K = 5

print(f"device={DEVICE}, d={D}, |A|={len(ALL_SCHEDULES)}, H={ROLLOUT_H}, T={COMMIT_T}")

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_target(d, seed=0):
    gen = torch.Generator().manual_seed(seed)
    return torch.randn(d, d, generator=gen) / math.sqrt(d)


def rand_matrix(d, seed):
    state = torch.cuda.get_rng_state()
    torch.cuda.manual_seed(seed)
    W = torch.randn(d, d, device="cuda") / math.sqrt(d)
    torch.cuda.set_rng_state(state)
    return W


def product(Ws):
    P = torch.eye(Ws[0].shape[0], device=Ws[0].device, dtype=Ws[0].dtype)
    for W in Ws:
        P = W @ P
    return P


def prefix_products(Ws):
    d = Ws[0].shape[0]
    prefixes = [torch.eye(d, device=Ws[0].device, dtype=Ws[0].dtype)]
    P = prefixes[0]
    for W in Ws:
        P = W @ P
        prefixes.append(P)
    return prefixes


def suffix_products(Ws):
    L, d = len(Ws), Ws[0].shape[0]
    suffixes = [None] * L
    S = torch.eye(d, device=Ws[0].device, dtype=Ws[0].dtype)
    for i in reversed(range(L)):
        suffixes[i] = S.clone()
        S = S @ Ws[i]
    return suffixes


def loss(Ws, A):
    E = product(Ws) - A
    return 0.5 * (E * E).sum()


def grads_chain(Ws, A):
    prefixes = prefix_products(Ws)
    suffixes = suffix_products(Ws)
    E = prefixes[-1] - A
    return [suffixes[i].T @ E @ prefixes[i].T for i in range(len(Ws))]



def spectralized_update(U, p=1.0, eps=1e-12):
    nrm = torch.linalg.norm(U, ord="fro")
    if nrm < eps:
        return U.clone()
    P, s, Qt = torch.linalg.svd(U, full_matrices=False)
    U_new = P @ torch.diag((s + eps) ** p) @ Qt
    return U_new * (nrm / (torch.linalg.norm(U_new, ord="fro") + eps))


def greedy_step(Ws, A, lr=1e-3, p=1.0):
    return [W + spectralized_update(-lr * G, p=p) for W, G in zip(Ws, grads_chain(Ws, A))]


def spectral_step(Ws, A, lr, schedule):
    return [W + spectralized_update(-lr * G, p=p) for W, G, p in zip(Ws, grads_chain(Ws, A), schedule)]


def rollout_schedule(Ws, A, schedule, lr, horizon):
    Ws_roll = [W.clone() for W in Ws]
    for _ in range(horizon):
        Ws_roll = spectral_step(Ws_roll, A, lr, schedule)
    return loss(Ws_roll, A).item()


def rollout_horizon(step, total_steps, h_max):
    """只用剩余步数截断，不引入训练阶段逻辑。"""
    return max(1, min(total_steps - step, h_max))


def select_best_schedule(Ws, A, lr, schedules, horizon):
    best_loss = float("inf")
    best_schedule = schedules[0]
    for schedule in schedules:
        final_loss = rollout_schedule(Ws, A, schedule, lr, horizon)
        if final_loss < best_loss:
            best_loss = final_loss
            best_schedule = schedule
    return best_schedule


def select_best_schedule_coarse_fine(Ws, A, lr, schedules, h_coarse, h_fine, top_k):
    ranked = []
    for schedule in schedules:
        ranked.append((rollout_schedule(Ws, A, schedule, lr, h_coarse), schedule))
    ranked.sort(key=lambda x: x[0])
    finalists = [s for _, s in ranked[:top_k]]
    return select_best_schedule(Ws, A, lr, finalists, h_fine)


def run_experiment(seed=0, d=64, L=3, steps=100, lr=2e-2,
                   schedules=None, rollout_h=15, commit_t=10,
                   use_coarse_fine=True, h_coarse=5, top_k=5,
                   print_every=50, verbose=True, dtype=torch.float64):
    if schedules is None:
        schedules = ALL_SCHEDULES

    set_seed(seed)
    A = make_target(d, seed=seed).to(device=DEVICE, dtype=dtype)
    Ws0 = [rand_matrix(d, seed + i).to(dtype=dtype) for i in range(L)]

    methods = {
        "sgd": [W.clone() for W in Ws0],
        "dp": [W.clone() for W in Ws0],
    }
    history = {
        "sgd": {"loss": []},
        "dp": {"loss": [], "spectral_schedule": []},
    }

    dp_schedule = None

    for t in range(steps):
        for name, Ws in methods.items():
            history[name]["loss"].append(loss(Ws, A).item())

        if verbose and print_every is not None and t % print_every == 0:
            msg = f"step {t:4d}"
            for name in methods:
                msg += f" | {name}: loss={history[name]['loss'][-1]:.4f}"
            print(msg)

        methods["sgd"] = greedy_step(methods["sgd"], A, lr=lr, p=1.0)

        if dp_schedule is None or t % commit_t == 0:
            h = rollout_horizon(t, steps, rollout_h)
            h_c = min(h, h_coarse)
            if use_coarse_fine and len(schedules) > top_k:
                dp_schedule = select_best_schedule_coarse_fine(
                    methods["dp"], A, lr, schedules, h_c, h, top_k,
                )
            else:
                dp_schedule = select_best_schedule(methods["dp"], A, lr, schedules, h)

        methods["dp"] = spectral_step(methods["dp"], A, lr, dp_schedule)
        history["dp"]["spectral_schedule"].append(tuple(dp_schedule))

    for name, Ws in methods.items():
        history[name]["loss"].append(loss(Ws, A).item())

    if verbose:
        print("\n=== Final training metrics ===")
        for name, Ws in methods.items():
            print(f"\n{name}")
            print("final loss          =", history[name]["loss"][-1])
            if name == "dp":
                print("spectral schedule counts:")
                for k, v in Counter(history["dp"]["spectral_schedule"]).items():
                    print("  ", k, ":", v)
    return history, methods, A

In [ ]:
def plot_training_curves(history, title="Training curves"):
  colors = {"sgd": "#243447", "dp": "#2F80ED"}

  fig, ax = plt.subplots(figsize=(7.2, 4.2))
  for name in history:
    ax.plot(history[name]["loss"], label=name, color=colors.get(name, "gray"), linewidth=2.0)
  ax.set_xlabel("step")
  ax.set_ylabel("loss")
  ax.set_title("Terminal loss")
  ax.legend(frameon=False)
  ax.grid(True, alpha=0.25)
  fig.suptitle(title)
  plt.tight_layout()
  plt.show()


In [ ]:
history, methods, A = run_experiment(
  seed=SEED, d=D, L=L, steps=STEPS, lr=LR,
  schedules=ALL_SCHEDULES,
  rollout_h=ROLLOUT_H,
  commit_t=COMMIT_T,
  use_coarse_fine=USE_COARSE_FINE,
  h_coarse=ROLLOUT_H_COARSE,
  top_k=ROLLOUT_TOP_K,
  print_every=50,
)

In [ ]:
plot_training_curves(history, title=f"SGD vs DP (|A|={len(ALL_SCHEDULES)}, H={ROLLOUT_H}, T={COMMIT_T})")


## Jacobian 价值估计

令当前误差为
$$
e=\operatorname{vec}(W_3W_2W_1-A).
$$
对于候选调度 $a=(p_1,p_2,p_3)$，先应用一步调度更新，得到新参数 $W^+(a)$ 和一步后的 loss：
$$
L^+(a)=\tfrac12\|e^+(a)\|_2^2.
$$
然后在 $W^+(a)$ 附近，对后续仍会训练的参数局部线性化误差。令 $J^+(a)$ 为对应 Jacobian，$K^+(a)=J^+(a)J^+(a)^\top$。当前误差方向上的有效收缩率为
$$
\lambda_{\mathrm{eff}}^+(a)
=
\frac{e^+(a)^\top K^+(a)e^+(a)}{e^+(a)^\top e^+(a)}
=
\frac{\|J^+(a)^\top e^+(a)\|_2^2}{\|e^+(a)\|_2^2}.
$$
最后一个形式正是代码采用的形式：$J^\top e$ 就是平方误差损失对可训练权重的梯度，因此我们可以通过普通梯度范数计算 $\lambda_{\mathrm{eff}}$，不需要显式构造完整的 Jacobian Gram 矩阵。

如果接下来 $H-1$ 步的局部几何变化较慢，则 horizon 为 $H$ 的预测 loss 为
$$
\widehat L_H(a)
=
L^+(a)\exp\left[-2\eta(H-1)\lambda_{\mathrm{eff}}^+(a)\right].
$$
这给出了一个具体的权衡判据。把候选动作 $a$ 和基准动作 $b$ 比较：
$$
\log\frac{L^+(a)}{L^+(b)}
<
2\eta(H-1)\left[\lambda_{\mathrm{eff}}^+(a)-\lambda_{\mathrm{eff}}^+(b)\right].
$$
左边是当前一步的 loss 代价，右边是预测的未来收敛率收益。当未来速率收益大于短期代价时，谱形状调整就是值得的。


## 三方法对比

下一个 cell 在同一问题上运行三种方法。对 `dp_jacobian`，打印出的决策行解释了为什么选择某个调度：`penalty_vs_sgd` 是相对 SGD 动作的当前 loss 代价，`rate_gain_vs_sgd` 是预测的未来速率优势，`pred_log_gain` 是二者的差。


In [ ]:
JAC_RATE_WEIGHT = 1.0
JAC_COMPARE_STEPS = 50


def jacobian_lambda_eff(Ws, A, trainable_from=0, eps=1e-12):
    # Return e^T J J^T e / e^T e without explicitly building J J^T.
    E = product(Ws) - A
    denom = (E * E).sum().item() + eps
    grads = grads_chain(Ws, A)[trainable_from:]
    numer = sum((G * G).sum().item() for G in grads)
    return numer / denom


def jacobian_schedule_score(Ws, A, schedule, lr, horizon, rate_weight=JAC_RATE_WEIGHT):
    Ws_next = spectral_step(Ws, A, lr, schedule)
    immediate = loss(Ws_next, A).item()
    lam = jacobian_lambda_eff(Ws_next, A)
    future_h = max(0, horizon - 1)
    predicted = immediate * math.exp(-2.0 * rate_weight * lr * future_h * lam)
    return predicted, immediate, lam


def select_best_schedule_jacobian(Ws, A, lr, schedules, horizon, rate_weight=JAC_RATE_WEIGHT):
    rows = []
    for schedule in schedules:
        predicted, immediate, lam = jacobian_schedule_score(
            Ws, A, schedule, lr, horizon, rate_weight=rate_weight,
        )
        rows.append((predicted, immediate, lam, tuple(schedule)))
    rows.sort(key=lambda x: x[0])
    return rows[0][3], rows


def tradeoff_certificate(best_row, baseline_row, lr, horizon):
    _, best_immediate, best_lam, best_schedule = best_row
    _, base_immediate, base_lam, base_schedule = baseline_row
    future_h = max(0, horizon - 1)
    immediate_penalty = math.log((best_immediate + 1e-12) / (base_immediate + 1e-12))
    rate_gain = 2.0 * lr * future_h * (best_lam - base_lam)
    predicted_log_gain = rate_gain - immediate_penalty
    return dict(
        chosen=best_schedule,
        baseline=base_schedule,
        immediate_penalty=immediate_penalty,
        rate_gain=rate_gain,
        predicted_log_gain=predicted_log_gain,
    )


def run_experiment_with_jacobian_value(
    seed=0, d=64, L=3, steps=50, lr=2e-2,
    schedules=None, rollout_h=15, commit_t=10,
    rate_weight=JAC_RATE_WEIGHT, verbose=True,
):
    if schedules is None:
        schedules = ALL_SCHEDULES

    set_seed(seed)
    A = make_target(d, seed=seed).to(device=DEVICE, dtype=torch.float64)
    Ws0 = [rand_matrix(d, seed + i).to(dtype=torch.float64) for i in range(L)]

    methods = {
        "sgd": [W.clone() for W in Ws0],
        "dp_rollout": [W.clone() for W in Ws0],
        "dp_jacobian": [W.clone() for W in Ws0],
    }
    history = {name: {"loss": []} for name in methods}
    history["dp_rollout"]["spectral_schedule"] = []
    history["dp_jacobian"]["spectral_schedule"] = []
    history["dp_jacobian"]["decision_rows"] = []

    rollout_schedule_current = None
    jacobian_schedule_current = None

    for t in range(steps):
        for name, Ws in methods.items():
            history[name]["loss"].append(loss(Ws, A).item())

        methods["sgd"] = greedy_step(methods["sgd"], A, lr=lr, p=1.0)

        if rollout_schedule_current is None or t % commit_t == 0:
            h = rollout_horizon(t, steps, rollout_h)
            rollout_schedule_current = select_best_schedule(
                methods["dp_rollout"], A, lr, schedules, h,
            )
        methods["dp_rollout"] = spectral_step(methods["dp_rollout"], A, lr, rollout_schedule_current)
        history["dp_rollout"]["spectral_schedule"].append(tuple(rollout_schedule_current))

        if jacobian_schedule_current is None or t % commit_t == 0:
            h = rollout_horizon(t, steps, rollout_h)
            jacobian_schedule_current, rows = select_best_schedule_jacobian(
                methods["dp_jacobian"], A, lr, schedules, h, rate_weight=rate_weight,
            )
            best = rows[0]
            sgd_row = next(row for row in rows if row[3] == (1.0,) * L)
            immediate_best = min(rows, key=lambda row: row[1])
            cert = tradeoff_certificate(best, sgd_row, lr, h)
            history["dp_jacobian"]["decision_rows"].append(dict(
                step=t,
                horizon=h,
                chosen=best[3],
                predicted=best[0],
                immediate=best[1],
                lambda_eff=best[2],
                immediate_best=immediate_best[3],
                sgd_predicted=sgd_row[0],
                sgd_immediate=sgd_row[1],
                sgd_lambda=sgd_row[2],
                immediate_penalty_vs_sgd=cert["immediate_penalty"],
                rate_gain_vs_sgd=cert["rate_gain"],
                predicted_log_gain_vs_sgd=cert["predicted_log_gain"],
            ))

        methods["dp_jacobian"] = spectral_step(methods["dp_jacobian"], A, lr, jacobian_schedule_current)
        history["dp_jacobian"]["spectral_schedule"].append(tuple(jacobian_schedule_current))

    for name, Ws in methods.items():
        history[name]["loss"].append(loss(Ws, A).item())

    if verbose:
        print("\n=== Jacobian-guided value comparison ===")
        for name in ["sgd", "dp_rollout", "dp_jacobian"]:
            print(f"{name:12s}: final loss={history[name]['loss'][-1]:.6f}")
        print("\nJacobian decisions:")
        for row in history["dp_jacobian"]["decision_rows"]:
            print(
                f"  step {row['step']:3d}, H={row['horizon']:2d}, chosen={row['chosen']}, "
                f"immediate={row['immediate']:.4f}, lambda={row['lambda_eff']:.4f}, "
                f"penalty_vs_sgd={row['immediate_penalty_vs_sgd']:+.4f}, "
                f"rate_gain_vs_sgd={row['rate_gain_vs_sgd']:+.4f}, "
                f"pred_log_gain={row['predicted_log_gain_vs_sgd']:+.4f}"
            )
        print("\nSchedule counts:")
        for k, v in Counter(history["dp_jacobian"]["spectral_schedule"]).items():
            print("  ", k, ":", v)

    return history, methods, A


jac_history, jac_methods, jac_A = run_experiment_with_jacobian_value(
    seed=SEED, d=D, L=L, steps=JAC_COMPARE_STEPS, lr=LR,
    schedules=ALL_SCHEDULES, rollout_h=ROLLOUT_H, commit_t=COMMIT_T,
    rate_weight=JAC_RATE_WEIGHT, verbose=True,
)


In [ ]:
def plot_jacobian_value_comparison(history, title="SGD vs Rollout-MPC vs Jacobian value"):
    colors = {"sgd": "#243447", "dp_rollout": "#2F80ED", "dp_jacobian": "#D946EF"}
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    for name in ["sgd", "dp_rollout", "dp_jacobian"]:
        ax.plot(history[name]["loss"], label=name, color=colors[name], linewidth=2.0)
    ax.set_xlabel("step")
    ax.set_ylabel("loss")
    ax.set_title("loss")
    ax.legend(frameon=False)
    ax.grid(True, alpha=0.25)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_jacobian_value_comparison(
    jac_history,
    title=f"50-step Jacobian-guided search (|A|={len(ALL_SCHEDULES)}, H={ROLLOUT_H}, T={COMMIT_T})",
)


## 快速评估给定固定调度

现在考虑一个实际问题：如果读者给出若干个候选谱模式，我们能否不完整训练每个候选方案，就快速判断它们谁更好？在这一节中，每个候选都是一个**固定调度**
$$
a=(p_1,p_2,p_3),
$$
也就是每个执行步都使用同一种谱模式。这里故意比完整的随时间变化 plan 更简单：目的是单独检验“短前缀 + Jacobian 价值估计”能否预测哪一种模式在更长 horizon 上更有希望。

一个固定调度的精确评估很直接但代价较高：从相同初始权重出发，完整执行 $T$ 步，然后比较终点 loss $L_T$。快速评估器用“前缀执行 + 局部价值估计”替代完整执行：

1. 只执行候选调度的前 $B<T$ 步，到达 $W_B(a)$。
2. 记录前缀 loss
$$
L_B(a)=\frac12\|F(W_B(a))-A\|_F^2,
$$
其中在这个 notebook 中 $F(W)=W_3W_2W_1$。
3. 在 $W_B(a)$ 处计算局部有效收缩率
$$
\lambda_{\mathrm{eff}}(W_B(a))
=
\frac{e_B^\top J_BJ_B^\top e_B}{e_B^\top e_B},
\qquad
e_B=\operatorname{vec}(F(W_B(a))-A).
$$
在线性网络情形下，这个量可以由各层梯度计算出来，因此不需要显式构造 $J_BJ_B^\top$。
4. 假设这个局部收缩率在剩余 $T-B$ 步中近似保持，并用 $\alpha\in(0,1]$ 做折扣：
$$
\widehat L_T(a)
=
L_B(a)\exp\{-2\alpha\eta (T-B)\lambda_{\mathrm{eff}}(W_B(a))\}.
$$

算法可以写成：

```text
输入：
  候选固定调度 a_1, ..., a_m
  目标 horizon T
  前缀预算 B < T
  学习率 eta
  几何折扣 alpha

对每个候选固定调度 a_i：
  重置到共同初始权重 W_0

  对 t = 0, ..., B-1：
    使用同一个调度 a_i 执行一步谱更新

  记录到达的状态 W_B(a_i)
  计算前缀 loss L_B(a_i)
  计算局部有效收缩率 lambda_eff(W_B(a_i))

  预测终点 loss：
    L_hat_T(a_i) = L_B(a_i) * exp(-2 * alpha * eta * (T-B) * lambda_eff(W_B(a_i)))

按照 L_hat_T(a_i) 给所有固定调度排序，越小越好。

仅用于验证的可选步骤：
  完整执行每个 a_i 共 T 步，并比较真实 L_T 与预测 L_hat_T。
```

最后按照 $\widehat L_T(a)$ 给候选调度排序。下面代码里的 `true_final` 只是 sanity check：它会完整执行每个调度，用来检查快速排序是否和真实终点 loss 一致。在真实搜索中，我们可以先用快速估计筛出有希望的调度，再把完整 rollout 预算花在少数候选上。

In [ ]:
import time


FIXED_TARGET_STEPS = 20
FIXED_PREFIX_STEPS = 1
FIXED_ALPHA = 0.05

PROPOSED_SCHEDULES = [
    (1.0, 1.0, 1.0),
    (0.5, 0.5, 0.5),
    (0.0, 0.0, 0.0),
    (1.0, 0.5, 1.0),
    (0.5, 0.0, 0.5),
    (0.0, 0.5, 0.0),
    (1.0, 0.0, 1.0),
    (0.0, 1.0, 0.0),
]


def clone_weights(Ws):
    return [W.clone() for W in Ws]


def make_problem(seed=SEED, d=D, L=L, dtype=torch.float64):
    set_seed(seed)
    A = make_target(d, seed=seed).to(device=DEVICE, dtype=dtype)
    Ws0 = [rand_matrix(d, seed + i).to(dtype=dtype) for i in range(L)]
    return Ws0, A


def run_fixed_schedule(Ws0, A, schedule, steps, lr=LR):
    Ws = clone_weights(Ws0)
    losses = [loss(Ws, A).item()]
    for _ in range(steps):
        Ws = spectral_step(Ws, A, lr, schedule)
        losses.append(loss(Ws, A).item())
    return Ws, losses


def rankdata(values):
    order = sorted(range(len(values)), key=lambda i: values[i])
    ranks = [0.0] * len(values)
    i = 0
    while i < len(values):
        j = i + 1
        while j < len(values) and values[order[j]] == values[order[i]]:
            j += 1
        avg = 0.5 * (i + j - 1)
        for k in range(i, j):
            ranks[order[k]] = avg
        i = j
    return ranks


def spearmanr(xs, ys):
    rx, ry = rankdata(xs), rankdata(ys)
    mx = sum(rx) / len(rx)
    my = sum(ry) / len(ry)
    vx = sum((x - mx) ** 2 for x in rx)
    vy = sum((y - my) ** 2 for y in ry)
    if vx == 0 or vy == 0:
        return float("nan")
    return sum((x - mx) * (y - my) for x, y in zip(rx, ry)) / math.sqrt(vx * vy)


def fast_fixed_schedule_evaluation(
    candidate_schedules=PROPOSED_SCHEDULES,
    target_steps=FIXED_TARGET_STEPS,
    prefix_steps=FIXED_PREFIX_STEPS,
    alpha=FIXED_ALPHA,
    seed=SEED,
    d=D,
    L=L,
    lr=LR,
    run_truth=True,
):
    Ws0, A = make_problem(seed=seed, d=d, L=L)
    prefix_steps = min(prefix_steps, target_steps)
    remaining = target_steps - prefix_steps

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    rows = []
    for schedule in candidate_schedules:
        Ws_prefix, prefix_losses = run_fixed_schedule(Ws0, A, schedule, prefix_steps, lr=lr)
        lam = jacobian_lambda_eff(Ws_prefix, A)
        predicted = prefix_losses[-1] * math.exp(-2.0 * alpha * lr * remaining * lam)
        one_step_loss = run_fixed_schedule(Ws0, A, schedule, 1, lr=lr)[1][-1]
        rows.append(dict(
            schedule=tuple(schedule),
            one_step_loss=one_step_loss,
            prefix_loss=prefix_losses[-1],
            lambda_eff=lam,
            predicted_final=predicted,
        ))
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    predictor_time = time.perf_counter() - t0

    truth_time = None
    if run_truth:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        for row in rows:
            _, full_losses = run_fixed_schedule(Ws0, A, row["schedule"], target_steps, lr=lr)
            row["true_final"] = full_losses[-1]
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        truth_time = time.perf_counter() - t1

    rows_by_pred = sorted(rows, key=lambda r: r["predicted_final"])
    rows_by_prefix = sorted(rows, key=lambda r: r["prefix_loss"])
    rows_by_one_step = sorted(rows, key=lambda r: r["one_step_loss"])
    print(f"target_steps={target_steps}, prefix_steps={prefix_steps}, alpha={alpha}")
    print(f"fast predictor time={predictor_time:.3f}s")
    if truth_time is not None:
        print(f"full fixed-schedule check time={truth_time:.3f}s, speedup={truth_time / predictor_time:.2f}x")
        rho_prefix = spearmanr([r["prefix_loss"] for r in rows], [r["true_final"] for r in rows])
        rho_pred = spearmanr([r["predicted_final"] for r in rows], [r["true_final"] for r in rows])
        print(f"Spearman(prefix-only, true)={rho_prefix:+.3f}")
        print(f"Spearman(predicted, true)={rho_pred:+.3f}")
    print(f"one-step greedy best: {rows_by_one_step[0]['schedule']}")
    print(f"prefix-only best: {rows_by_prefix[0]['schedule']}")
    print(f"prefix-Jacobian best: {rows_by_pred[0]['schedule']}")

    header = "rank | schedule        | one-step | prefix | lambda_eff | predicted"
    if run_truth:
        header += " | true"
    print("\n" + header)
    for k, row in enumerate(rows_by_pred, 1):
        line = (
            f"{k:4d} | {str(row['schedule']):15s} | "
            f"{row['one_step_loss']:.4f} | {row['prefix_loss']:.4f} | "
            f"{row['lambda_eff']:.4f} | {row['predicted_final']:.4f}"
        )
        if run_truth:
            line += f" | {row['true_final']:.4f}"
        print(line)
    return rows, dict(predictor_time=predictor_time, truth_time=truth_time)


fixed_rows, fixed_timing = fast_fixed_schedule_evaluation()


In [ ]:
def plot_fixed_schedule_evaluation(rows):
    ordered = sorted(rows, key=lambda r: r["predicted_final"])
    labels = ["".join(str(int(2 * p)) for p in r["schedule"]) for r in ordered]
    x = list(range(len(ordered)))
    width = 0.36

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
    axes[0].bar([i - width / 2 for i in x], [r["predicted_final"] for r in ordered],
                width=width, label="predicted", color="#2F80ED")
    if "true_final" in ordered[0]:
        axes[0].bar([i + width / 2 for i in x], [r["true_final"] for r in ordered],
                    width=width, label="true", color="#F2994A")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels)
    axes[0].set_xlabel("schedule code: 2=1.0, 1=0.5, 0=0.0")
    axes[0].set_ylabel("final loss")
    axes[0].set_title("Predicted vs true terminal loss")
    axes[0].legend(frameon=False)
    axes[0].grid(True, axis="y", alpha=0.25)

    if "true_final" in ordered[0]:
        axes[1].scatter([r["predicted_final"] for r in ordered],
                        [r["true_final"] for r in ordered],
                        s=60, color="#27AE60")
        for label, row in zip(labels, ordered):
            axes[1].annotate(label, (row["predicted_final"], row["true_final"]),
                             textcoords="offset points", xytext=(4, 4), fontsize=9)
        axes[1].set_xlabel("predicted final loss")
        axes[1].set_ylabel("true final loss")
        axes[1].set_title("Ranking sanity check")
        axes[1].grid(True, alpha=0.25)
    else:
        axes[1].axis("off")

    plt.tight_layout()
    plt.show()


plot_fixed_schedule_evaluation(fixed_rows)


## 如何解读结果

Jacobian 引导的策略通常一开始会选择更激进的谱形状调整，因为预测的未来速率收益足以抵消一步 loss 代价。随着训练推进，速率优势变小，所选调度会逐渐接近 SGD 动作。

下面的固定调度筛选实验被设置成展示 Jacobian 项为什么有用。当前缀预算 $B=1$ 时，prefix-only 筛选本质上就是单步贪心信号：它只问哪一个调度在一步更新后的 loss 最小。在这次实验中，这个贪心准则偏好接近 SGD 的调度 $(1,1,1)$。

但在更长 horizon 上，真实最优固定调度并不是它。前缀 Jacobian 估计同样只执行一步，但还会评估到达状态处的局部收缩率。这个未来收缩项改变了排序，并选择 $(0.5,0.5,0.5)$，与完整 $T$ 步检查一致。

折扣系数 $\\alpha$ 很重要。如果不折扣，单个局部 Jacobian 可能高估那些“有利几何”只能短暂保持的调度。使用较小的折扣后，预测仍然便宜，但可以看得比当前一步 loss 更远。